In [4]:
# Notebook: nb_silver_market_data_clean
# Purpose: Clean JSE and SARB data and add DQ scores (Steps 15 & 16)

from pyspark.sql import functions as F
from datetime import datetime

print(" Starting Silver Layer Market Data Cleaning...")
print("=" * 60)


StatementMeta(, d79a6b84-4ee1-48ed-a47e-61c601472963, 6, Finished, Available, Finished, False)

 Starting Silver Layer Market Data Cleaning...


In [5]:
# PART 1: Clean JSE Listings
print("\n Processing JSE Listings...")
jse_bronze = spark.table("bronze_jse_listings")

# Clean and select key columns
jse_silver = jse_bronze.select(
    F.col("JSE_Code"),
    F.upper(F.trim(F.col("CompanyName"))).alias("CompanyName"),
    F.upper(F.trim(F.col("Sector"))).alias("Sector"),
    F.col("SharePrice_ZAR").cast("double"),
    F.col("Market")
).dropDuplicates(["JSE_Code"])

# Add Data Quality Score & Timestamp
jse_silver = jse_silver.withColumn("dq_score", F.lit(100)) \
                       .withColumn("silver_load_timestamp", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

# Write to Silver
jse_silver.write.format("delta").mode("overwrite").saveAsTable("silver_jse_listings")
print(f" Created silver_jse_listings with {jse_silver.count()} rows.")


StatementMeta(, d79a6b84-4ee1-48ed-a47e-61c601472963, 7, Finished, Available, Finished, False)


 Processing JSE Listings...
 Created silver_jse_listings with 50 rows.


In [6]:
# PART 2: Clean SARB Bank Stats
print("\n Processing SARB Bank Stats...")
sarb_bronze = spark.table("bronze_sarb_stats")

# Clean and select key columns
# Note: Removed QuarterEndDate as it's not in our synthetic Bronze data
sarb_silver = sarb_bronze.select(
    F.col("BankName"),
    F.col("BankCode"),
    F.col("Quarter"),
    F.col("TotalAssets_Billions_ZAR").cast("double"),
    F.col("CapitalAdequacyRatio_pct").cast("double"),
    F.col("LiquidityCoverageRatio_pct").cast("double"),
    F.col("NonPerformingLoans_pct").cast("double"),
    F.col("ReturnOnEquity_pct").cast("double")
).dropDuplicates(["BankCode", "Quarter"])

# Add Data Quality Score & Timestamp
sarb_silver = sarb_silver.withColumn("dq_score", F.lit(100)) \
                         .withColumn("silver_load_timestamp", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

# Write to Silver
sarb_silver.write.format("delta").mode("overwrite").saveAsTable("silver_sarb_banks")
print(f" Created silver_sarb_banks with {sarb_silver.count()} rows.")

print("\n" + "=" * 60)
print(" Silver Market Data Cleaning Complete!")
print("Tables created: silver_jse_listings, silver_sarb_banks")

StatementMeta(, d79a6b84-4ee1-48ed-a47e-61c601472963, 8, Finished, Available, Finished, False)


 Processing SARB Bank Stats...
 Created silver_sarb_banks with 40 rows.

 Silver Market Data Cleaning Complete!
Tables created: silver_jse_listings, silver_sarb_banks
